# 无监督盲 FIR 尖峰均衡提取 Click 多径结构

目标记录：`010 | Voyage_Ori_01 / PulseTrain_014 / Click_008`

仅使用单通道观测波形，不使用已知发射信号、参考 Click 或从观测信号构造的源模板。假设

$$x[n]=s[n]*h[n]+v[n],$$

未知源波形 $s[n]$ 相对短且平稳，多径响应 $h[n]$ 是稀疏的。寻找 FIR 均衡器 $g[n]$，使 $z[n]=g[n]*x[n]$ 的尖峰性（归一化四阶矩）最大，以 $z[n]$ 作为稀疏到达响应的估计。

为减少非凸优化的局部最优影响，运行多个确定性与随机初值，将高目标值结果按主到达对齐，从共识包络中报告时延、相对强度和跨初值支持率。

> 单通道盲均衡存在整体时移、尺度、符号和源/信道分解不唯一性；因此结果表达相对多径结构，不是唯一的绝对信道冲激响应。

In [ ]:
from pathlib import Path
import csv
import json
import warnings

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.linalg import cho_factor, cho_solve
from scipy.signal import butter, sosfiltfilt, find_peaks, hilbert, windows, correlate, correlation_lags
from numpy.lib.stride_tricks import sliding_window_view

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, total=None, desc=None, **kwargs):
        return iterable if iterable is not None else None

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 125

PROJECT_ROOT = Path(r"D:\Project_Github\Indo-Pacific-humpback-dolphin")
INPUT_WAV = PROJECT_ROOT / r"00_Data\Positive_Clicks_3500\Voyage_Ori_01\PulseTrain_014\Click_008\Pulse_008.wav"
OUTPUT_DIR = PROJECT_ROOT / r"18_多径提取算法\Blind_FIR_Equalization_010_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 预处理和盲均衡参数
LOWCUT_HZ = 20_000.0
HIGHCUT_HZ = 190_000.0
FILTER_ORDER = 6
EQUALIZER_LENGTH_MS = 0.35
N_STARTS = 10
ITERATIONS = 70
RIDGE_RATIO = 2e-4
DAMPING = 0.30
CONSENSUS_RUNS = 5
RANDOM_SEED = 20260813

# 多径检测参数
DIRECT_SEARCH_RADIUS_MS = 0.18
MAX_DELAY_MS = 3.20
MIN_PATH_DELAY_MS = 0.070
MIN_PATH_DISTANCE_MS = 0.060
ENVELOPE_SMOOTH_US = 8.0
PEAK_RELATIVE_THRESHOLD = 0.035
PEAK_PROMINENCE_RATIO = 0.018
ROBUST_SIGMA = 7.0
MAX_PATHS = 10
SUPPORT_TOLERANCE_US = 25.0

# 重构解释率参数
SOURCE_LENGTH_MS = 0.30
SOURCE_RIDGE_RATIO = 1e-3
RECONSTRUCTION_MAX_LAG_MS = 0.20
SPARSE_SIGN_RADIUS_US = 12.0

# 蒙特卡洛路径置信度参数
MONTE_CARLO_TRIALS = 100
MONTE_CARLO_NOISE_SCALE = 1.0
MONTE_CARLO_NOISE_BLOCK_US = 50.0
MONTE_CARLO_MATCH_TOLERANCE_US = 35.0
MONTE_CARLO_SEED = 20260814

print(f"输入：{INPUT_WAV}")
print(f"输出：{OUTPUT_DIR}")
assert INPUT_WAV.exists(), f"找不到输入 WAV：{INPUT_WAV}"

## 1. 读取与预处理

保留完整 10 ms 波形，去直流后执行 20–190 kHz 零相位带通。Tukey 窗仅用于减轻截断边界不连续对盲均衡的影响。原波形包络的全局最大值只用于统一各盲解的整体时移，不作为均衡器的训练模板。

In [ ]:
fs, raw = wavfile.read(INPUT_WAV)
if raw.ndim == 2:
    raw = raw.mean(axis=1)
if np.issubdtype(raw.dtype, np.integer):
    scale = max(abs(np.iinfo(raw.dtype).min), np.iinfo(raw.dtype).max)
    x_raw = raw.astype(np.float64) / scale
else:
    x_raw = raw.astype(np.float64)

x_raw -= np.mean(x_raw)
nyquist = fs / 2.0
if not (0 < LOWCUT_HZ < HIGHCUT_HZ < nyquist):
    raise ValueError(f"带通范围必须低于 Nyquist 频率 {nyquist:,.0f} Hz")

sos = butter(FILTER_ORDER, [LOWCUT_HZ / nyquist, HIGHCUT_HZ / nyquist], btype="bandpass", output="sos")
x = sosfiltfilt(sos, x_raw)
x *= windows.tukey(len(x), alpha=0.02)
x /= np.max(np.abs(x)) + 1e-12

t_ms = np.arange(len(x)) / fs * 1e3
raw_envelope = np.abs(hilbert(x))
raw_anchor_sample = int(np.argmax(raw_envelope))
raw_anchor_ms = raw_anchor_sample / fs * 1e3

print(f"采样率：{fs:,} Hz")
print(f"样本数：{len(x):,}（{len(x) / fs * 1e3:.3f} ms）")
print(f"用于盲解时移对齐的观测主峰：{raw_anchor_ms:.4f} ms")

fig, axes = plt.subplots(2, 1, figsize=(11, 6), constrained_layout=True)
axes[0].plot(t_ms, x_raw / (np.max(np.abs(x_raw)) + 1e-12), color="0.68", lw=0.8, label="原始")
axes[0].plot(t_ms, x, color="#176B87", lw=0.9, label="20–190 kHz 带通")
axes[0].axvline(raw_anchor_ms, color="#188977", ls="--", lw=1.0, label="对齐锚点")
axes[0].set(xlabel="时间 (ms)", ylabel="归一化幅值", title="观测 Click 波形")
axes[0].legend(loc="upper right")
axes[0].grid(alpha=0.2)

freq_khz = np.fft.rfftfreq(len(x), d=1 / fs) / 1e3
spec = np.abs(np.fft.rfft(x))
spec_db = 20 * np.log10(spec / (np.max(spec) + 1e-12) + 1e-12)
axes[1].plot(freq_khz, spec_db, color="#B2533E", lw=0.9)
axes[1].axvspan(LOWCUT_HZ / 1e3, HIGHCUT_HZ / 1e3, color="#B2533E", alpha=0.08)
axes[1].set(xlim=(0, nyquist / 1e3), ylim=(-90, 3), xlabel="频率 (kHz)", ylabel="相对幅度 (dB)", title="预处理信号频谱")
axes[1].grid(alpha=0.2)
plt.show()

## 2. 无监督盲 FIR 尖峰均衡

将观测信号的滞后样本组成 $X$，均衡输出为 $z=Xg$。固定点更新为

$$g \leftarrow (R_{xx}+\lambda I)^{-1}X^T\tilde z^3,\qquad R_{xx}=X^TX/N,$$

其中 $\tilde z$ 已去均值并标准化。每次更新都在 $R_{xx}$ 度量下归一化，使输出方差受约束，因而实际上最大化输出尖峰性。所有初值和迭代共用一个“盲 FIR 均衡总体进度”进度条。

In [ ]:
def excess_kurtosis(values):
    values = np.asarray(values, dtype=float)
    values = values - np.mean(values)
    variance = np.mean(values ** 2) + 1e-15
    return float(np.mean(values ** 4) / variance ** 2 - 3.0)


def metric_normalize(fir, covariance):
    return fir / (np.sqrt(float(fir @ covariance @ fir)) + 1e-12)


def make_initializers(length, count, rng):
    starts = []
    # 单位冲激只是 FIR 系数初值，不是信号模板
    for tap in np.linspace(0, length - 1, min(4, count), dtype=int):
        g = np.zeros(length)
        g[tap] = 1.0
        starts.append(g)
    while len(starts) < count:
        g = rng.normal(size=length)
        g *= windows.tukey(length, alpha=0.35)
        starts.append(g)
    return starts


def blind_fir_spiking_equalization(signal, sample_rate, *, show_progress=True, random_seed=RANDOM_SEED):
    length = max(15, int(round(EQUALIZER_LENGTH_MS * 1e-3 * sample_rate)))
    X = sliding_window_view(signal, length)[:, ::-1].copy()
    n_rows = X.shape[0]
    covariance = (X.T @ X) / n_rows
    ridge = RIDGE_RATIO * max(np.trace(covariance) / length, 1e-12)
    covariance_reg = covariance + ridge * np.eye(length)
    factor = cho_factor(covariance_reg, lower=True, check_finite=False)
    rng = np.random.default_rng(random_seed)
    initializers = make_initializers(length, N_STARTS, rng)
    results = []

    progress = tqdm(total=N_STARTS * ITERATIONS, desc="盲 FIR 均衡总体进度", unit="次", disable=not show_progress)
    for start_index, initial in enumerate(initializers):
        g = metric_normalize(initial, covariance_reg)
        history = []
        for _ in range(ITERATIONS):
            z = X @ g
            z_centered = z - np.mean(z)
            z_standard = z_centered / (np.sqrt(np.mean(z_centered ** 2)) + 1e-12)
            rhs = (X.T @ (z_standard ** 3)) / n_rows
            candidate = cho_solve(factor, rhs, check_finite=False)
            candidate = metric_normalize(candidate, covariance_reg)
            if np.dot(candidate, g) < 0:
                candidate = -candidate
            g = metric_normalize((1.0 - DAMPING) * g + DAMPING * candidate, covariance_reg)
            history.append(excess_kurtosis(X @ g))
            progress.update(1)
        z = X @ g
        z -= np.median(z)
        results.append({
            "start": start_index + 1,
            "filter": g,
            "output": z,
            "history": np.asarray(history),
            "objective": excess_kurtosis(z),
        })
    progress.close()
    results.sort(key=lambda item: item["objective"], reverse=True)
    return results, X, covariance_reg


results, X, covariance_reg = blind_fir_spiking_equalization(x, fs)
equalizer_length = X.shape[1]
output_samples = np.arange(X.shape[0]) + equalizer_length - 1
output_t_ms = output_samples / fs * 1e3
objectives = np.asarray([item["objective"] for item in results])

print(f"FIR 均衡器长度：{equalizer_length} 点（{equalizer_length / fs * 1e3:.4f} ms）")
print("各初值最终超额峰度（由高到低）：")
print(np.array2string(objectives, precision=3, separator=", "))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for rank, item in enumerate(results):
    axes[0].plot(np.arange(1, ITERATIONS + 1), item["history"], lw=0.8, alpha=0.45 if rank else 1.0, label="最优解" if rank == 0 else None)
axes[0].set(xlabel="迭代次数", ylabel="超额峰度", title="多初值收敛曲线")
axes[0].legend()
axes[0].grid(alpha=0.2)
axes[1].bar(np.arange(1, N_STARTS + 1), objectives, color=["#188977"] + ["#7A5195"] * (N_STARTS - 1))
axes[1].set(xlabel="目标值排名", ylabel="最终超额峰度", title="盲均衡解的目标值")
axes[1].grid(axis="y", alpha=0.2)
plt.show()

## 3. 多初值对齐、共识与到达检测

盲均衡的整体时移本来就不可辨识。对每个高目标值输出，在观测主峰附近寻找它的最强均衡尖峰，并将该尖峰对齐到观测主峰。对齐后的归一化包络中位数是共识响应；一条径的“支持率”是在 ±25 μs 容差内出现局部峰的高目标值解比例。

In [ ]:
def smooth(values, length):
    length = max(1, int(length))
    return np.convolve(values, np.ones(length) / length, mode="same")


def shift_with_zeros(values, shift):
    shifted = np.zeros_like(values)
    if shift >= 0:
        shifted[shift:] = values[:len(values) - shift] if shift else values
    else:
        shifted[:shift] = values[-shift:]
    return shifted


smooth_samples = max(1, int(round(ENVELOPE_SMOOTH_US * 1e-6 * fs)))
target_index = int(np.argmin(np.abs(output_samples - raw_anchor_sample)))
search_radius = max(1, int(round(DIRECT_SEARCH_RADIUS_MS * 1e-3 * fs)))
left = max(0, target_index - search_radius)
right = min(len(output_samples), target_index + search_radius + 1)

aligned_envelopes = []
aligned_outputs = []
for item in results[:CONSENSUS_RUNS]:
    z = item["output"] / (np.max(np.abs(item["output"])) + 1e-12)
    env = smooth(np.abs(hilbert(z)), smooth_samples)
    local_peak = left + int(np.argmax(env[left:right]))
    shift = target_index - local_peak
    aligned_outputs.append(shift_with_zeros(z, shift))
    aligned = shift_with_zeros(env, shift)
    aligned_envelopes.append(aligned / (np.max(aligned) + 1e-12))

aligned_envelopes = np.asarray(aligned_envelopes)
aligned_outputs = np.asarray(aligned_outputs)
consensus_envelope = np.median(aligned_envelopes, axis=0)
consensus_envelope /= np.max(consensus_envelope) + 1e-12
best_output = aligned_outputs[0] / (np.max(np.abs(aligned_outputs[0])) + 1e-12)

direct_index = target_index
max_delay_samples = int(round(MAX_DELAY_MS * 1e-3 * fs))
min_delay_samples = int(round(MIN_PATH_DELAY_MS * 1e-3 * fs))
end_index = min(len(consensus_envelope), direct_index + max_delay_samples + 1)
detection_slice = consensus_envelope[direct_index:end_index]
median = np.median(detection_slice)
mad = np.median(np.abs(detection_slice - median)) + 1e-12
height_threshold = max(PEAK_RELATIVE_THRESHOLD, median + ROBUST_SIGMA * 1.4826 * mad)
distance_samples = max(1, int(round(MIN_PATH_DISTANCE_MS * 1e-3 * fs)))

local_peaks, _ = find_peaks(
    detection_slice,
    height=height_threshold,
    prominence=PEAK_PROMINENCE_RATIO,
    distance=distance_samples,
)
peak_indices = direct_index + local_peaks
peak_indices = peak_indices[peak_indices >= direct_index + min_delay_samples]
if len(peak_indices) > MAX_PATHS - 1:
    strongest = np.argsort(consensus_envelope[peak_indices])[-(MAX_PATHS - 1):]
    peak_indices = np.sort(peak_indices[strongest])
peak_indices = np.concatenate([[direct_index], peak_indices.astype(int)])

support_tolerance = max(1, int(round(SUPPORT_TOLERANCE_US * 1e-6 * fs)))
run_peak_sets = []
for env in aligned_envelopes:
    peaks, _ = find_peaks(env, height=max(0.02, 0.5 * height_threshold), distance=distance_samples)
    run_peak_sets.append(peaks)

direct_amplitude = float(consensus_envelope[direct_index])
paths = []
for number, index in enumerate(peak_indices):
    amplitude = float(consensus_envelope[index])
    support = np.mean([np.any(np.abs(peaks - index) <= support_tolerance) for peaks in run_peak_sets])
    paths.append({
        "path": "direct" if number == 0 else f"multipath_{number}",
        "arrival_sample": int(output_samples[index]),
        "arrival_time_ms": float(output_t_ms[index]),
        "relative_delay_ms": float((index - direct_index) / fs * 1e3),
        "relative_amplitude": float(amplitude / (direct_amplitude + 1e-12)),
        "relative_level_db": float(20 * np.log10((amplitude + 1e-12) / (direct_amplitude + 1e-12))),
        "multistart_support": float(support),
    })

print(f"共识包络检测门限：{height_threshold:.4f}")
print(f"检测到 {len(paths)} 条显著到达（含主到达）")
print(f"{'路径':<14}{'相对时延(ms)':>15}{'相对强度(dB)':>16}{'初值支持率':>13}")
for item in paths:
    print(f"{item['path']:<14}{item['relative_delay_ms']:>15.4f}{item['relative_level_db']:>16.2f}{item['multistart_support']:>12.0%}")

## 4. 基于100次背景噪声扰动的路径置信度

从主到达前、远离主波包的区间估计背景噪声。每次试验按50 us分块有放回重采样噪声，加回观测波形，然后重新执行完整的多初值盲 FIR 均衡和峰值检测。扰动路径与原始路径在 ±35 us 容差内做一对一最近匹配。

置信度定义为该路径在100次试验中的重复检出比例。这是条件于当前噪声模型、参数与匹配容差的蒙特卡洛重复检出率，不是路径真实存在的后验概率。主到达是固定的对齐锚点，因此不估计其置信度。

In [ ]:
def block_bootstrap_noise(noise_pool, output_length, block_length, rng):
    noise_pool = np.asarray(noise_pool, dtype=float) - np.mean(noise_pool)
    block_length = min(max(2, int(block_length)), len(noise_pool))
    pieces, remaining = [], output_length
    while remaining > 0:
        start = int(rng.integers(0, len(noise_pool) - block_length + 1))
        block = noise_pool[start:start + block_length]
        pieces.append(block[:remaining])
        remaining -= len(block)
    return np.concatenate(pieces)


def detect_delays_from_equalizer_results(trial_results, sample_rate, fixed_anchor_sample):
    eq_length = len(trial_results[0]["filter"])
    trial_output_samples = np.arange(len(trial_results[0]["output"])) + eq_length - 1
    target = int(np.argmin(np.abs(trial_output_samples - fixed_anchor_sample)))
    radius = max(1, int(round(DIRECT_SEARCH_RADIUS_MS * 1e-3 * sample_rate)))
    search_left, search_right = max(0, target - radius), min(len(trial_output_samples), target + radius + 1)
    trial_envelopes = []
    for item in trial_results[:CONSENSUS_RUNS]:
        output = item["output"] / (np.max(np.abs(item["output"])) + 1e-12)
        envelope = smooth(np.abs(hilbert(output)), smooth_samples)
        local = search_left + int(np.argmax(envelope[search_left:search_right]))
        aligned = shift_with_zeros(envelope, target - local)
        trial_envelopes.append(aligned / (np.max(aligned) + 1e-12))
    consensus = np.median(np.asarray(trial_envelopes), axis=0)
    consensus /= np.max(consensus) + 1e-12
    trial_end = min(len(consensus), target + int(round(MAX_DELAY_MS * 1e-3 * sample_rate)) + 1)
    segment = consensus[target:trial_end]
    segment_median = np.median(segment)
    segment_mad = np.median(np.abs(segment - segment_median)) + 1e-12
    threshold = max(PEAK_RELATIVE_THRESHOLD, segment_median + ROBUST_SIGMA * 1.4826 * segment_mad)
    trial_peaks, _ = find_peaks(segment, height=threshold, prominence=PEAK_PROMINENCE_RATIO, distance=max(1, int(round(MIN_PATH_DISTANCE_MS * 1e-3 * sample_rate))))
    minimum_delay = int(round(MIN_PATH_DELAY_MS * 1e-3 * sample_rate))
    trial_peaks = trial_peaks[trial_peaks >= minimum_delay]
    if len(trial_peaks) > MAX_PATHS - 1:
        strongest = np.argsort(consensus[target + trial_peaks])[-(MAX_PATHS - 1):]
        trial_peaks = np.sort(trial_peaks[strongest])
    return trial_peaks.astype(int), float(threshold)


def one_to_one_delay_matches(reference_delays, detected_delays, tolerance_samples):
    pairs = []
    for reference_index, reference in enumerate(reference_delays):
        for detected_index, detected in enumerate(detected_delays):
            difference = abs(int(detected) - int(reference))
            if difference <= tolerance_samples:
                pairs.append((difference, reference_index, detected_index))
    pairs.sort()
    used_references, used_detections, matches = set(), set(), {}
    for _, reference_index, detected_index in pairs:
        if reference_index not in used_references and detected_index not in used_detections:
            matches[reference_index] = int(detected_delays[detected_index])
            used_references.add(reference_index)
            used_detections.add(detected_index)
    return matches


noise_start = int(round(0.25e-3 * fs))
noise_end = raw_anchor_sample - int(round(0.60e-3 * fs))
if noise_end - noise_start < int(round(1.0e-3 * fs)):
    raise ValueError("主到达前的可用背景噪声段不足1 ms")
noise_pool = x[noise_start:noise_end]
noise_rms = float(np.sqrt(np.mean((noise_pool - np.mean(noise_pool)) ** 2)))
noise_block_samples = max(2, int(round(MONTE_CARLO_NOISE_BLOCK_US * 1e-6 * fs)))
match_tolerance_samples = max(1, int(round(MONTE_CARLO_MATCH_TOLERANCE_US * 1e-6 * fs)))
reference_delay_samples = np.asarray([int(round(item["relative_delay_ms"] * 1e-3 * fs)) for item in paths[1:]], dtype=int)
matched_delay_samples = [[] for _ in reference_delay_samples]
trial_peak_counts, trial_thresholds = [], []
rng_mc = np.random.default_rng(MONTE_CARLO_SEED)

progress_mc = tqdm(total=MONTE_CARLO_TRIALS, desc="多径置信度蒙特卡洛总体进度", unit="次")
for trial in range(MONTE_CARLO_TRIALS):
    added_noise = block_bootstrap_noise(noise_pool, len(x), noise_block_samples, rng_mc)
    perturbed = x + MONTE_CARLO_NOISE_SCALE * added_noise
    perturbed /= np.max(np.abs(perturbed)) + 1e-12
    trial_results, _, _ = blind_fir_spiking_equalization(perturbed, fs, show_progress=False, random_seed=RANDOM_SEED)
    detected_delay_samples, trial_threshold = detect_delays_from_equalizer_results(trial_results, fs, raw_anchor_sample)
    matches = one_to_one_delay_matches(reference_delay_samples, detected_delay_samples, match_tolerance_samples)
    for reference_index, detected in matches.items():
        matched_delay_samples[reference_index].append(detected)
    trial_peak_counts.append(len(detected_delay_samples))
    trial_thresholds.append(trial_threshold)
    progress_mc.update(1)
progress_mc.close()

confidence_rows = [{"path": "direct", "reference_delay_ms": 0.0, "monte_carlo_confidence": None, "detections": None, "trials": MONTE_CARLO_TRIALS, "delay_median_ms": 0.0, "delay_ci95_low_ms": 0.0, "delay_ci95_high_ms": 0.0, "status": "fixed alignment anchor; confidence not estimated"}]
paths[0].update({"monte_carlo_confidence": None, "delay_median_ms_mc": 0.0, "delay_ci95_low_ms_mc": 0.0, "delay_ci95_high_ms_mc": 0.0})
for index, samples in enumerate(matched_delay_samples, start=1):
    delay_values_ms = np.asarray(samples, dtype=float) / fs * 1e3
    confidence = len(samples) / MONTE_CARLO_TRIALS
    median_delay = float(np.median(delay_values_ms)) if len(samples) else np.nan
    ci_low, ci_high = (np.quantile(delay_values_ms, [0.025, 0.975]).tolist() if len(samples) else [np.nan, np.nan])
    paths[index].update({"monte_carlo_confidence": float(confidence), "delay_median_ms_mc": median_delay, "delay_ci95_low_ms_mc": float(ci_low), "delay_ci95_high_ms_mc": float(ci_high)})
    confidence_rows.append({"path": paths[index]["path"], "reference_delay_ms": paths[index]["relative_delay_ms"], "monte_carlo_confidence": float(confidence), "detections": len(samples), "trials": MONTE_CARLO_TRIALS, "delay_median_ms": median_delay, "delay_ci95_low_ms": float(ci_low), "delay_ci95_high_ms": float(ci_high), "status": "estimated"})

print(f"背景噪声段：{noise_start / fs * 1e3:.3f}–{noise_end / fs * 1e3:.3f} ms，RMS={noise_rms:.6f}")
print(f"噪声扰动：{MONTE_CARLO_NOISE_SCALE:.2f} x 背景噪声；匹配容差：±{MONTE_CARLO_MATCH_TOLERANCE_US:.1f} us")
print(f"{'路径':<14}{'置信度':>10}{'检出次数':>12}{'时延中位数(ms)':>18}{'95%时延区间(ms)':>24}")
print(f"{'direct':<14}{'--':>10}{'fixed':>12}{0.0:>18.4f}{'[0, 0]':>24}")
for row in confidence_rows[1:]:
    interval = f"[{row['delay_ci95_low_ms']:.4f}, {row['delay_ci95_high_ms']:.4f}]"
    print(f"{row['path']:<14}{row['monte_carlo_confidence']:>9.1%}{str(row['detections']) + '/100':>12}{row['delay_median_ms']:>18.4f}{interval:>24}")

confidence_csv_path = OUTPUT_DIR / "blind_fir_equalization_path_confidence.csv"
with confidence_csv_path.open("w", newline="", encoding="utf-8-sig") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(confidence_rows[0].keys()))
    writer.writeheader()
    writer.writerows(confidence_rows)
confidence_fig, confidence_ax = plt.subplots(figsize=(10.5, 4.8), constrained_layout=True)
labels = [row["path"].replace("multipath_", "多径 ") for row in confidence_rows[1:]]
confidence_values = np.asarray([row["monte_carlo_confidence"] for row in confidence_rows[1:]])
bars = confidence_ax.bar(labels, 100 * confidence_values, color="#176B87", alpha=0.88)
for bar, row in zip(bars, confidence_rows[1:]):
    confidence_ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5, f"{100 * row['monte_carlo_confidence']:.0f}%\n{row['detections']}/100", ha="center", va="bottom", fontsize=9)
confidence_ax.set(ylim=(0, 110), ylabel="蒙特卡洛重复检出率 (%)", title="各多径在100次背景噪声扰动下的置信度")
confidence_ax.grid(axis="y", alpha=0.20)
confidence_path = OUTPUT_DIR / "blind_fir_equalization_path_confidence.png"
confidence_fig.savefig(confidence_path, dpi=220, bbox_inches="tight")
plt.show()
print(f"路径置信度表：{confidence_csv_path}")
print(f"路径置信度图：{confidence_path}")

## 5. 波形重构与样本内解释率

分别以：（1）对齐后的最优完整均衡输出；（2）仅保留已检测到达的带符号稀疏响应，作为两种候选信道响应 $\hat h$。在不引入外部模板的前提下，用正则化最小二乘从同一条观测波形中估计长度固定的未知短源波形 $\hat s$：

$$\hat s=\arg\min_s\|x-Hs\|_2^2+\lambda\|s\|_2^2.$$

对候选重构波形在限定范围内拟合整体时移、增益和偏置，计算

$$R^2=1-\frac{\sum_n(x[n]-\hat x[n])^2}{\sum_n(x[n]-\bar x)^2}.$$

> 源波形和评分使用了同一条 Click，所以这是样本内描述性解释率，会比独立模板或样本外评估更乐观。

In [ ]:
def shifted_copy(values, shift):
    result = np.zeros_like(values)
    if shift >= 0:
        if shift < len(values):
            result[shift:] = values[:len(values) - shift]
    else:
        if -shift < len(values):
            result[:shift] = values[-shift:]
    return result


def estimate_source_and_candidate(observed, response, source_length, ridge_ratio):
    # 用以 0 为中心的短时延基表示未知源，不使用源模板
    offsets = np.arange(source_length) - source_length // 2
    H = np.column_stack([shifted_copy(response, int(offset)) for offset in offsets])
    gram = H.T @ H
    ridge = ridge_ratio * max(np.trace(gram) / source_length, 1e-12)
    source = np.linalg.solve(gram + ridge * np.eye(source_length), H.T @ observed)
    candidate = H @ source
    return source, candidate, offsets, ridge


def align_scale_and_score(observed, candidate, max_lag_samples):
    observed_centered = observed - np.mean(observed)
    candidate_centered = candidate - np.mean(candidate)
    cross = correlate(observed_centered, candidate_centered, mode="full", method="fft")
    lags = correlation_lags(len(observed), len(candidate), mode="full")
    allowed = np.abs(lags) <= max_lag_samples
    allowed_indices = np.flatnonzero(allowed)
    lag = int(lags[allowed_indices[np.argmax(np.abs(cross[allowed]))]])
    aligned = shifted_copy(candidate, lag)
    design = np.column_stack([aligned, np.ones(len(aligned))])
    gain, offset = np.linalg.lstsq(design, observed, rcond=None)[0]
    fitted = gain * aligned + offset
    residual = observed - fitted
    sse = float(np.sum(residual ** 2))
    sst = float(np.sum((observed - np.mean(observed)) ** 2))
    r_squared = float(1.0 - sse / (sst + 1e-15))
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    return fitted, r_squared, rmse, lag, float(gain), float(offset)


# 将均衡输出放回原 WAV 的绝对样本坐标
full_response = np.zeros(len(x))
full_response[output_samples] = best_output
full_response /= np.max(np.abs(full_response)) + 1e-12

# 在每个共识包络峰附近，从最优带符号输出取局部最强样本
sparse_response = np.zeros(len(x))
sign_radius = max(1, int(round(SPARSE_SIGN_RADIUS_US * 1e-6 * fs)))
sparse_impulse_samples = []
for index in peak_indices:
    left_local = max(0, int(index) - sign_radius)
    right_local = min(len(best_output), int(index) + sign_radius + 1)
    local_index = left_local + int(np.argmax(np.abs(best_output[left_local:right_local])))
    absolute_sample = int(output_samples[local_index])
    sparse_response[absolute_sample] = best_output[local_index]
    sparse_impulse_samples.append(absolute_sample)
sparse_response /= np.max(np.abs(sparse_response)) + 1e-12

source_length = max(9, int(round(SOURCE_LENGTH_MS * 1e-3 * fs)))
if source_length % 2 == 0:
    source_length += 1
max_lag_samples = max(1, int(round(RECONSTRUCTION_MAX_LAG_MS * 1e-3 * fs)))

source_full, candidate_full, source_offsets, source_ridge_full = estimate_source_and_candidate(
    x, full_response, source_length, SOURCE_RIDGE_RATIO
)
y_hat_full, r2_full, rmse_full, lag_full, gain_full, offset_full = align_scale_and_score(
    x, candidate_full, max_lag_samples
)

source_sparse, candidate_sparse, _, source_ridge_sparse = estimate_source_and_candidate(
    x, sparse_response, source_length, SOURCE_RIDGE_RATIO
)
y_hat_sparse, r2_sparse, rmse_sparse, lag_sparse, gain_sparse, offset_sparse = align_scale_and_score(
    x, candidate_sparse, max_lag_samples
)

print(f"估计源波形长度：{source_length} 点（{source_length / fs * 1e3:.4f} ms）")
print(f"完整均衡响应重构：R^2={r2_full:.4f}（{100 * r2_full:.2f}%），RMSE={rmse_full:.5f}，对齐 {lag_full} 点")
print(f"检测峰稀疏响应重构：R^2={r2_sparse:.4f}（{100 * r2_sparse:.2f}%），RMSE={rmse_sparse:.5f}，对齐 {lag_sparse} 点")

reconstruction_fig, reconstruction_axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
reconstruction_axes[0].plot(source_offsets / fs * 1e6, source_full / (np.max(np.abs(source_full)) + 1e-12), color="#176B87", lw=1.0, label="完整响应对应源估计")
reconstruction_axes[0].plot(source_offsets / fs * 1e6, source_sparse / (np.max(np.abs(source_sparse)) + 1e-12), color="#C73E1D", lw=1.0, alpha=0.85, label="稀疏响应对应源估计")
reconstruction_axes[0].set(xlabel="相对时间 (us)", ylabel="归一化幅值", title="正则化最小二乘估计的未知短源波形")
reconstruction_axes[0].legend(loc="upper right")
reconstruction_axes[0].grid(alpha=0.18)

reconstruction_axes[1].plot(t_ms, x, color="0.25", lw=0.9, label="观测波形")
reconstruction_axes[1].plot(t_ms, y_hat_full, color="#176B87", lw=0.9, alpha=0.9, label=f"完整响应重构，R^2={r2_full:.3f}")
reconstruction_axes[1].set(xlabel="时间 (ms)", ylabel="幅值", title="完整盲均衡响应的样本内重构")
reconstruction_axes[1].legend(loc="upper right")
reconstruction_axes[1].grid(alpha=0.18)

reconstruction_axes[2].plot(t_ms, x, color="0.25", lw=0.9, label="观测波形")
reconstruction_axes[2].plot(t_ms, y_hat_sparse, color="#C73E1D", lw=0.9, alpha=0.9, label=f"检测峰稀疏重构，R^2={r2_sparse:.3f}")
reconstruction_axes[2].set(xlabel="时间 (ms)", ylabel="幅值", title="已检测多径峰的样本内稀疏重构")
reconstruction_axes[2].legend(loc="upper right")
reconstruction_axes[2].grid(alpha=0.18)
for ax in reconstruction_axes[1:]:
    ax.set_xlim(max(0, raw_anchor_ms - 0.45), min(t_ms[-1], raw_anchor_ms + MAX_DELAY_MS + 0.25))

reconstruction_path = OUTPUT_DIR / "blind_fir_equalization_reconstruction.png"
reconstruction_fig.savefig(reconstruction_path, dpi=220, bbox_inches="tight")
plt.show()

## 6. 单独绘制估计的原始 Click：$s_{\mathrm{sparse}}$

将“已检测多径峰的稀疏响应”对应的正则化最小二乘源估计 `source_sparse` 定义为 $s_{\mathrm{sparse}}$。图中时域波形按最大绝对幅值归一化；频谱以自身最大值为 0 dB。时间零点是源估计的参考中心，不是绝对发声时刻。

In [ ]:
s_sparse = source_sparse.copy()
s_sparse_normalized = s_sparse / (np.max(np.abs(s_sparse)) + 1e-12)
s_sparse_time_us = source_offsets / fs * 1e6
s_sparse_envelope = np.abs(hilbert(s_sparse_normalized))

nfft_source = 1 << int(np.ceil(np.log2(max(1024, 8 * len(s_sparse)))))
s_sparse_frequency_khz = np.fft.rfftfreq(nfft_source, d=1 / fs) / 1e3
s_sparse_spectrum = np.abs(np.fft.rfft(s_sparse_normalized * windows.tukey(len(s_sparse_normalized), alpha=0.15), n=nfft_source))
s_sparse_spectrum_db = 20 * np.log10(s_sparse_spectrum / (np.max(s_sparse_spectrum) + 1e-12) + 1e-12)

source_fig, source_axes = plt.subplots(2, 1, figsize=(11, 7), constrained_layout=True)
source_axes[0].plot(s_sparse_time_us, s_sparse_normalized, color="#176B87", lw=1.35, label=r"$s_{\mathrm{sparse}}$")
source_axes[0].plot(s_sparse_time_us, s_sparse_envelope, color="#C73E1D", lw=1.0, alpha=0.8, label="Hilbert 包络")
source_axes[0].plot(s_sparse_time_us, -s_sparse_envelope, color="#C73E1D", lw=1.0, alpha=0.8)
source_axes[0].axvline(0, color="0.45", ls=":", lw=0.9)
source_axes[0].set(xlabel="相对时间 (us)", ylabel="归一化幅值", title=r"估计的原始 Click 时域波形 $s_{\mathrm{sparse}}$")
source_axes[0].legend(loc="upper right")
source_axes[0].grid(alpha=0.20)

source_axes[1].plot(s_sparse_frequency_khz, s_sparse_spectrum_db, color="#7A5195", lw=1.2)
source_axes[1].axvspan(LOWCUT_HZ / 1e3, HIGHCUT_HZ / 1e3, color="#7A5195", alpha=0.08, label="20–190 kHz 分析带")
source_axes[1].set(xlim=(0, nyquist / 1e3), ylim=(-80, 3), xlabel="频率 (kHz)", ylabel="相对幅度 (dB)", title=r"$s_{\mathrm{sparse}}$ 的幅度频谱")
source_axes[1].legend(loc="upper right")
source_axes[1].grid(alpha=0.20)

source_path = OUTPUT_DIR / "estimated_original_click_s_sparse.png"
source_fig.savefig(source_path, dpi=220, bbox_inches="tight")
plt.show()

source_csv_path = OUTPUT_DIR / "estimated_original_click_s_sparse.csv"
with source_csv_path.open("w", newline="", encoding="utf-8-sig") as handle:
    writer = csv.writer(handle)
    writer.writerow(["sample_offset", "relative_time_us", "amplitude", "normalized_amplitude"])
    writer.writerows(zip(source_offsets.astype(int), s_sparse_time_us, s_sparse, s_sparse_normalized))

print(f"s_sparse 长度：{len(s_sparse)} 点（{len(s_sparse) / fs * 1e3:.4f} ms）")
print(f"s_sparse 图：{source_path}")
print(f"s_sparse 数值：{source_csv_path}")

## 7. 结果图与导出

综合图展示观测波形、最优单次均衡输出、高目标值解的包络与共识，以及最终多径时延结构。另外单独导出适合引用的多径结构图、CSV 路径表和 JSON 元数据。

In [ ]:
colors = ["#188977"] + ["#C73E1D"] * (len(paths) - 1)
fig, axes = plt.subplots(4, 1, figsize=(12, 12), constrained_layout=True)

axes[0].plot(t_ms, x, color="#176B87", lw=0.9)
axes[0].set(ylabel="幅值", title="(a) 20–190 kHz 带通后的观测 Click")
axes[0].grid(alpha=0.18)

axes[1].plot(output_t_ms, best_output, color="0.42", lw=0.75)
axes[1].plot(output_t_ms, smooth(np.abs(hilbert(best_output)), smooth_samples), color="#7A5195", lw=1.0, alpha=0.9)
axes[1].set(ylabel="归一化幅值", title="(b) 最优目标值的盲 FIR 尖峰均衡输出")
axes[1].grid(alpha=0.18)

for rank, env in enumerate(aligned_envelopes):
    axes[2].plot(output_t_ms, env, color="0.72", lw=0.65, alpha=0.55, label="高目标值单次解" if rank == 0 else None)
axes[2].plot(output_t_ms, consensus_envelope, color="#B32134", lw=1.35, label="对齐后共识包络")
axes[2].axhline(height_threshold, color="#B32134", ls=":", lw=1.0, label="检测门限")
for color, index in zip(colors, peak_indices):
    axes[2].axvline(output_t_ms[index], color=color, ls="--", lw=0.9, alpha=0.85)
    axes[2].scatter(output_t_ms[index], consensus_envelope[index], color=color, edgecolor="white", s=38, zorder=5)
axes[2].set(ylabel="归一化包络", title="(c) 多初值对齐共识与到达检测")
axes[2].legend(loc="upper right", ncol=3, fontsize=8)
axes[2].grid(alpha=0.18)

delays = np.asarray([item["relative_delay_ms"] for item in paths])
levels = np.asarray([item["relative_level_db"] for item in paths])
supports = np.asarray([item["multistart_support"] for item in paths])
markerline, stemlines, baseline = axes[3].stem(delays, levels, linefmt="#C73E1D", markerfmt="o", basefmt=" ")
plt.setp(markerline, markersize=7, markerfacecolor="#C73E1D", markeredgecolor="white")
plt.setp(stemlines, linewidth=2, alpha=0.78)
axes[3].scatter(delays[0], levels[0], s=85, color="#188977", edgecolor="white", zorder=5)
for number, item in enumerate(paths):
    label = "主到达" if number == 0 else f"多径 {number}\n{item['relative_delay_ms']:.3f} ms\n支持 {item['multistart_support']:.0%}"
    offset = (-58, 8) if number == 2 else ((8, 26) if number == 3 else (5, 7))
    axes[3].annotate(label, (delays[number], levels[number]), xytext=offset, textcoords="offset points", fontsize=8)
axes[3].axhline(0, color="0.45", ls=":", lw=0.9)
axes[3].set(xlabel="相对主到达时延 (ms)", ylabel="相对强度 (dB)", title="(d) 010 | Voyage_Ori_01 / PulseTrain_014 / Click_008 多径结构")
axes[3].grid(alpha=0.2)
axes[3].margins(y=0.22)

for ax in axes[:3]:
    ax.set_xlim(max(0, raw_anchor_ms - 0.45), min(t_ms[-1], raw_anchor_ms + MAX_DELAY_MS + 0.25))
    ax.set_xlabel("时间 (ms)")

overview_path = OUTPUT_DIR / "blind_fir_equalization_overview.png"
fig.savefig(overview_path, dpi=220, bbox_inches="tight")
plt.show()

structure_fig, structure_ax = plt.subplots(figsize=(10.5, 4.8), constrained_layout=True)
markerline, stemlines, baseline = structure_ax.stem(delays, levels, linefmt="#B32134", markerfmt="o", basefmt=" ")
plt.setp(markerline, markersize=8, markerfacecolor="#B32134", markeredgecolor="white")
plt.setp(stemlines, linewidth=2.2, alpha=0.82)
structure_ax.scatter(delays[0], levels[0], s=95, color="#188977", edgecolor="white", zorder=5)
for number, item in enumerate(paths):
    text = "主到达" if number == 0 else f"多径 {number}\n{item['relative_delay_ms']:.3f} ms, {item['relative_level_db']:.1f} dB\n支持率 {item['multistart_support']:.0%}"
    offset = (-76, 8) if number == 2 else ((10, 30) if number == 3 else (6, 8))
    structure_ax.annotate(text, (delays[number], levels[number]), xytext=offset, textcoords="offset points", fontsize=9)
structure_ax.axhline(0, color="0.45", ls=":", lw=0.9)
structure_ax.set(xlabel="相对主到达时延 (ms)", ylabel="相对强度 (dB)", title="无监督盲 FIR 尖峰均衡：多径时延结构")
structure_ax.grid(alpha=0.22)
structure_ax.margins(y=0.24)
structure_path = OUTPUT_DIR / "blind_fir_equalization_path_structure.png"
structure_fig.savefig(structure_path, dpi=220, bbox_inches="tight")
plt.show()

csv_path = OUTPUT_DIR / "blind_fir_equalization_paths.csv"
with csv_path.open("w", newline="", encoding="utf-8-sig") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(paths[0].keys()))
    writer.writeheader()
    writer.writerows(paths)

metadata = {
    "record": "010 | Voyage_Ori_01 / PulseTrain_014 / Click_008",
    "input_wav": str(INPUT_WAV),
    "sample_rate_hz": int(fs),
    "method": "unsupervised multistart blind FIR spiking equalization (MED fixed-point objective)",
    "uses_signal_template": False,
    "parameters": {
        "bandpass_hz": [LOWCUT_HZ, HIGHCUT_HZ],
        "equalizer_length_ms": EQUALIZER_LENGTH_MS,
        "n_starts": N_STARTS,
        "iterations_per_start": ITERATIONS,
        "ridge_ratio": RIDGE_RATIO,
        "damping": DAMPING,
        "consensus_runs": CONSENSUS_RUNS,
        "maximum_delay_ms": MAX_DELAY_MS,
        "minimum_path_delay_ms": MIN_PATH_DELAY_MS,
        "peak_relative_threshold": PEAK_RELATIVE_THRESHOLD,
        "peak_prominence_ratio": PEAK_PROMINENCE_RATIO,
        "support_tolerance_us": SUPPORT_TOLERANCE_US,
        "source_length_ms": SOURCE_LENGTH_MS,
        "source_ridge_ratio": SOURCE_RIDGE_RATIO,
        "reconstruction_max_lag_ms": RECONSTRUCTION_MAX_LAG_MS,
        "sparse_sign_radius_us": SPARSE_SIGN_RADIUS_US,
        "monte_carlo_trials": MONTE_CARLO_TRIALS,
        "monte_carlo_noise_scale": MONTE_CARLO_NOISE_SCALE,
        "monte_carlo_noise_block_us": MONTE_CARLO_NOISE_BLOCK_US,
        "monte_carlo_match_tolerance_us": MONTE_CARLO_MATCH_TOLERANCE_US,
        "monte_carlo_seed": MONTE_CARLO_SEED,
        "random_seed": RANDOM_SEED,
    },
    "ranked_final_excess_kurtosis": objectives.tolist(),
    "detection_threshold": float(height_threshold),
    "monte_carlo_confidence": {
        "definition": "repeat detection rate under block-bootstrap background-noise perturbations; not posterior existence probability",
        "noise_pool_start_ms": float(noise_start / fs * 1e3),
        "noise_pool_end_ms": float(noise_end / fs * 1e3),
        "noise_pool_rms": float(noise_rms),
        "mean_detected_paths_per_trial": float(np.mean(trial_peak_counts)),
        "mean_detection_threshold": float(np.mean(trial_thresholds)),
        "path_results": confidence_rows,
    },
    "reconstruction": {
        "scope": "in-sample descriptive reconstruction; source estimated from the same observed click",
        "source_length_samples": int(source_length),
        "full_response_r_squared": float(r2_full),
        "sparse_detected_paths_r_squared": float(r2_sparse),
        "full_response_rmse": float(rmse_full),
        "sparse_detected_paths_rmse": float(rmse_sparse),
        "full_alignment_lag_samples": int(lag_full),
        "sparse_alignment_lag_samples": int(lag_sparse),
        "full_fitted_gain": float(gain_full),
        "sparse_fitted_gain": float(gain_sparse),
    },
    "paths": paths,
    "identifiability_note": "Single-channel blind equalization has scale, sign, overall-delay and source/channel factorization ambiguities; paths are relative structural estimates.",
}
json_path = OUTPUT_DIR / "blind_fir_equalization_metadata.json"
json_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"路径置信度图：{confidence_path}")
print(f"路径置信度表：{confidence_csv_path}")
print(f"重构图：{reconstruction_path}")
print(f"综合图：{overview_path}")
print(f"多径结构图：{structure_path}")
print(f"路径表：{csv_path}")
print(f"参数与结果：{json_path}")

## 8. 结果解释边界

- `0 ms` 表示用观测主峰统一对齐后的主到达，不是 WAV 起点。
- 相对强度来自盲均衡共识包络，不等于未均衡原波形的绝对声压级。
- 支持率反映非凸优化在不同初值下的重现性，不是统计显著性或真实概率。
- 蒙特卡洛置信度是当前背景噪声分块重采样模型下的重复检出率，依赖噪声强度与 ±35 us 匹配容差，不是物理路径存在概率。
- FIR 长度、峰值门限和最小径间隔都会影响近间隔到达的分辨。用于批量统计前，应在独立人工标注集上固定参数。
- 完整响应与检测峰稀疏响应的重构解释率均为样本内指标；未知源波形由同一条 Click 估计，因此不能解释为样本外预测能力。
- $s_{\mathrm{sparse}}$ 的时间零点不是绝对发声时刻；波形主包络相对零点的偏移属于盲分解的整体时移不确定性。
- 单个 Click 且没有参考信号时，不能仅依据高峰度证明每个尖峰都是物理反射径；结果宜与原波形、相邻 Click 的重复时延或传播几何共同核验。